### ***CNN Implemenation CIFAR10 Dataset***

In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 

# Load Dataset 
import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# 🔥 Train transform (with augmentation)
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

# ✅ Test transform (clean)
test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

train_set = CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_set = CIFAR10(root="./data", train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

100%|██████████| 170M/170M [04:37<00:00, 614kB/s]  
d:\Generative AI\Genai_Series\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


### ***Let's Build Our CNN***

In [3]:
import torch.nn as nn

class CNN(nn.Module): 
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),   # 🔥 helps generalization
            nn.Linear(256, 10)
        )

    def forward(self, x): 
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x)

In [4]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [5]:
epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}")

Epoch [1/30], Loss: 1135.9318
Epoch [2/30], Loss: 822.3185
Epoch [3/30], Loss: 701.9590
Epoch [4/30], Loss: 634.7550
Epoch [5/30], Loss: 581.2314
Epoch [6/30], Loss: 546.2166
Epoch [7/30], Loss: 506.7005
Epoch [8/30], Loss: 483.5557
Epoch [9/30], Loss: 463.8307
Epoch [10/30], Loss: 443.7052
Epoch [11/30], Loss: 427.3300
Epoch [12/30], Loss: 410.5402
Epoch [13/30], Loss: 398.4551
Epoch [14/30], Loss: 385.2810
Epoch [15/30], Loss: 376.5373
Epoch [16/30], Loss: 363.3670
Epoch [17/30], Loss: 353.9222
Epoch [18/30], Loss: 345.2879
Epoch [19/30], Loss: 336.7087
Epoch [20/30], Loss: 331.1383
Epoch [21/30], Loss: 323.4048
Epoch [22/30], Loss: 317.0476
Epoch [23/30], Loss: 311.4066
Epoch [24/30], Loss: 309.5132
Epoch [25/30], Loss: 301.2208
Epoch [26/30], Loss: 293.8320
Epoch [27/30], Loss: 291.5979
Epoch [28/30], Loss: 285.5931
Epoch [29/30], Loss: 281.6829
Epoch [30/30], Loss: 276.0011


In [6]:
# Evaluate Our CNN 
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy: {100 * correct / total:.2f}%")

Accuracy: 85.01%


In [7]:
import torch

# after training
torch.save(model.state_dict(), "cifar10.pth")